In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import tkinter as tk
from tkinter import filedialog
from clase_reportes import ReportClass
rc = ReportClass()


In [2]:
ruta = False

In [ ]:
if ruta: # Si se proporciona una ruta, úsala
    ruta = Path(ruta)
    ruta_kits = ruta / 'data' / 'kits.xlsx'
    df_kits = pd.read_excel(ruta_kits)
else:
    ruta = rc.validar_ruta()
    ruta_kits = ruta / 'data' / 'kits.xlsx'
    df_kits = pd.read_excel(ruta_kits)



ruta_base =ruta / 'file' / 'BASE_VENTAS.xlsx'


# Cargar el archivo de Excel
df_ventas = pd.read_excel(ruta_base)  # Reemplaza con la ruta de tu archivo

# Verifica si falta incluir kit en el archivo kits.xlsx
df_explosion_prueba = pd.merge(df_ventas, df_kits, left_on="PRODUCTO", right_on="KIT", indicator=True, how='left')

productos_con_kit = [
    producto for producto in df_explosion_prueba[df_explosion_prueba['_merge']=='left_only']['PRODUCTO_x'].unique()
    if 'KIT' in str(producto)
]
print(f"Productos con 'KIT' sin correspondencia en df_kits: {productos_con_kit}")


# Unir df_ventas con df_kits para explotar los kits
df_explosion = pd.merge(df_ventas, df_kits, left_on="PRODUCTO", right_on="KIT")

# Agregar una columna para indicar el origen (kit o individual)
df_explosion["ORIGEN"] = "KIT"

# Calcular las cantidades de productos
df_explosion["CANTIDAD_PRODUCTO"] = df_explosion["CANTIDAD"]


# Calcular el valor por producto en los kits
# df_explosion["VALOR_POR_PRODUCTO"] = df_explosion["TOTAL($)"] / df_explosion.groupby("KIT")["PRODUCTO_x"].transform("count")
conteo_facturas = df_explosion.groupby(['PRODUCTO_x','NUMERO_FACTURA'])['NUMERO_FACTURA'].transform('count')
df_explosion["VALOR_POR_PRODUCTO"] = df_explosion['TOTAL($)'] / conteo_facturas


# Filtrar productos individuales
df_ventas_individuales = df_ventas[~df_ventas["PRODUCTO"].str.startswith(("[PCNKIT","[TNGKIT","[B8KIT"))].reset_index(drop=True)
df_ventas_individuales["ORIGEN"] = "INDIVIDUAL"

# Calcular las cantidades de productos
df_ventas_individuales["CANTIDAD_PRODUCTO"] = df_ventas_individuales["CANTIDAD"]
df_ventas_individuales["VALOR_POR_PRODUCTO"] = df_ventas_individuales['TOTAL($)'] 


Productos con 'KIT' sin correspondencia en df_kits: ['[PCNKIT28] KIT 40+', '[PCNKIT26] KIT CLEAN LOOK', '[PCNKIT29] KIT ANTI-FRIZZ LISOS Y ONDULADOS', '[PCNKIT27] KIT KERATINA REPAIR', '[PCNKIT24] KIT PROTECCIÓN DEPORTIVA', '[PCNKIT23] KIT PROTECCIÓN DE VERANO', '[PCNKIT30] KIT ANTI-FRIZZ RIZOS', '[PCNKIT25] KIT SECADO Y ONDAS SOÑADAS', '[PCNKIT22] KIT PRE DECOLORACIÓN', '[PCN15] KIT HYDRO CLEAN HC']


In [4]:
df_ventas_individuales.columns

Index(['NUMERO_FACTURA', 'FECHA_FACTURA', 'AÑO', 'MES', 'DIA', 'CLIENTE',
       'IDENTIFICACION_CLIENTE', 'CATEGORÍA', 'PRODUCTO', 'CANTIDAD', 'TOTAL',
       'TASA_CAMBIO', 'TRM', 'TOTAL($)', 'TELEFONO', 'EMAIL', 'PAIS', 'CIUDAD',
       'CIUDAD_CORREGIDA', 'DEPARTAMENTO', 'EQUIPO_VENTAS', 'REFERENCIA',
       'ZONA', 'ASESOR COMERCIAL', 'ORIGEN', 'CANTIDAD_PRODUCTO',
       'VALOR_POR_PRODUCTO'],
      dtype='object')

In [5]:
df_ventas_individuales['PRODUCTO_y'] = df_ventas_individuales['PRODUCTO'] 

In [6]:

df_final_completo = pd.concat([df_explosion, df_ventas_individuales], ignore_index=True)


In [9]:
df_final_completo.columns

Index(['NUMERO_FACTURA', 'FECHA_FACTURA', 'CLIENTE', 'CATEGORÍA', 'PRODUCTO_y',
       'CANTIDAD_PRODUCTO', 'VALOR_POR_PRODUCTO', 'ORIGEN'],
      dtype='object')

In [ ]:
# Angelica: Reordenar columnas para que coincidan con el formato solicitado
df_final_completo = df_final_completo[['NUMERO_FACTURA','FECHA_FACTURA','CLIENTE','CATEGORÍA','PRODUCTO_y','CANTIDAD_PRODUCTO', 'VALOR_POR_PRODUCTO','ORIGEN']]

In [ ]:
# Daniel: Reordenar columnas para que coincidan con el formato solicitado
# df_final_completo = df_final_completo[['NUMERO_FACTURA','FECHA_FACTURA','CLIENTE','CATEGORÍA','PRODUCTO_y','CANTIDAD_PRODUCTO', 'VALOR_POR_PRODUCTO','ORIGEN']]

In [8]:
df_final_completo.to_excel(r"C:\Users\Dataa\Desktop\dani.xlsx")